# Machine Learning (Logistische Regression, Decision Tree und K-Nearest-Neighbors)

## Quelle des Datensets

### Das Datenset ist ein öffentlich verfügbares Datenset für Forschungszwecke von Cortez et al. (2009).

Quelle: P. Cortez, A. Cerdeira, F. Almeida, T. Matos and J. Reis (2009).   
Modeling wine preferences by data mining from physicochemical properties.  
Decision Support Systems, 47(4), 547–553.   
Link zum Artikel: https://doi.org/10.1016/j.dss.2009.05.016  
Link zum Datenset: https://archive.ics.uci.edu/dataset/186/wine+quality  

## Fragestellung: Kann man anhand der chemischen Eigenschaften vorhersagen, ob ein Wein eine hohe Qualität besitzt?

### Setup (Libraries)

In [54]:
# Datenverarbeitung
import pandas as pd
import numpy as np

# ML-Train-Test-Split und Skalierung
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Klassifikationsmodelle
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

# Evaluationsmetriken
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

### CSV Datei einlesen

In [12]:
df = pd.read_csv("winequality_red_white.csv", sep=";") # Erstellt in lab08

## Überblick über Spalte Quality

In [20]:
df["quality"].describe() # Qualität wird in einer Skala von 0 - 9 angegeben

count    6497.000000
mean        5.818378
std         0.873255
min         3.000000
25%         5.000000
50%         6.000000
75%         6.000000
max         9.000000
Name: quality, dtype: float64

## Klassen bilden

In [58]:
# Festlegen der Klassen:
# good_wine = 1, wenn quality >= 7 (also hohe Qualität)
# good_wine = 0, sonst (also niedrige/mittlere Qualität)
df['good_wine'] = (df['quality'] >= 7).astype(int)

# Prüfen der Klassenverteilung (prozentual)
df['good_wine'].value_counts(normalize=True)

good_wine
0    0.803448
1    0.196552
Name: proportion, dtype: float64

In [66]:
# Kategorische Variable 'type' in numerische 0/1-Codierung umwandeln
# rot = 0, weiß = 1 dadurch Zahlen, statt Strings für die ML-Modelle
df['type'] = df['type'].map({'red': 0, 'white': 1})

## Alle Inputs (Spalten mit chemischen Eigenschaften) und Zielvariable definieren

In [ ]:
# Liste aller Inputs (chemische Eigenschaften)
features = [
    'fixed acidity',
    'volatile acidity',
    'citric acid',
    'residual sugar',
    'chlorides',
    'free sulfur dioxide',
    'total sulfur dioxide',
    'density',
    'pH',
    'sulphates',
    'alcohol',
    'type'
]

# Alle Spalten, die als Input genommen werden
X = df[features]

# Zielvariable
y = df['good_wine']

## Split von Trainings- und Testdaten einstellen, um Modelle am Ende zu prüfen

In [70]:
# Aufteilen in Trainings- und Testdaten
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,     # 20% Testdaten
    stratify=y,        # Klassenverteilung gleich halten
    random_state=42    # für Reproduzierbarkeit
)

## Features skalieren für Logistische Regression und KNN

In [73]:
# StandardScaler: Features auf Mittelwert 0 und Standardabweichung 1 bringen
scaler = StandardScaler()

# Nur Trainingsdaten fitten
X_train_scaled = scaler.fit_transform(X_train)

# Testdaten transformieren mit gleichen Parametern
X_test_scaled = scaler.transform(X_test)

## Logistische Regression

In [80]:
# Lineares, interpretierbares Klassifikationsmodell
logreg = LogisticRegression(max_iter=1000)

# Modell auf Trainingsdaten fitten
logreg.fit(X_train_scaled, y_train)

# Vorhersagen für Testdaten
y_pred_logreg = logreg.predict(X_test_scaled)

## Decision Tree

In [83]:
# Nichtlineares Modell, einfache Interpretierbarkeit
tree = DecisionTreeClassifier(max_depth=5, random_state=42)

# Modell auf Trainingsdaten fitten
tree.fit(X_train, y_train)

# Vorhersagen für Testdaten
y_pred_tree = tree.predict(X_test)

## K-Nearest Neighbors

In [86]:
# Instanzbasiertes Modell, skalenabhängig
knn = KNeighborsClassifier(n_neighbors=5)

# Modell auf skalierten Trainingsdaten fitten
knn.fit(X_train_scaled, y_train)

# Vorhersagen für Testdaten
y_pred_knn = knn.predict(X_test_scaled)

## Evaluation der Modelle

In [89]:
# Hilfsfunktion zur Ausgabe wichtiger Metriken
def evaluate(y_true, y_pred, model_name):
    print(f"\n--- {model_name} ---")
    print("Accuracy :", accuracy_score(y_true, y_pred))         # Gesamtgenauigkeit
    print("Precision:", precision_score(y_true, y_pred))       # Anteil korrekt vorhergesagter guter Weine
    print("Recall   :", recall_score(y_true, y_pred))          # Anteil korrekt gefundener guter Weine
    print("F1-score :", f1_score(y_true, y_pred))              # Harmonisches Mittel von Precision und Recall
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred)) # Matrix: TP, FP, TN, FN

evaluate(y_test, y_pred_logreg, "Logistic Regression")
evaluate(y_test, y_pred_tree, "Decision Tree")
evaluate(y_test, y_pred_knn, "KNN")


--- Logistic Regression ---
Accuracy : 0.8223076923076923
Precision: 0.6146788990825688
Recall   : 0.26171875
F1-score : 0.36712328767123287
Confusion matrix:
 [[1002   42]
 [ 189   67]]

--- Decision Tree ---
Accuracy : 0.8315384615384616
Precision: 0.648
Recall   : 0.31640625
F1-score : 0.4251968503937008
Confusion matrix:
 [[1000   44]
 [ 175   81]]

--- KNN ---
Accuracy : 0.8323076923076923
Precision: 0.5922330097087378
Recall   : 0.4765625
F1-score : 0.5281385281385281
Confusion matrix:
 [[960  84]
 [134 122]]


## Bewertung der Ergebnisse

- Accuracy liegt bei allen Modellen bei ca. 82–83%, guter Wert aber eventuell nicht ganz aussagekräftig,
  weil die meisten Weine schlecht sind (Klassen-Ungleichgewicht).
- Logistishe Regression: Stabil und gut interpretierbar, erkennt aber nur wenige gute Weine (niedriger Recall).
- Decision Tree: Erkennt etwas mehr gute Weine als bei der logistischen Regression.
- KNN: Beste Balance zwischen Precision und Recall, erkennt fast die Hälfte der guten Weine.
- Hauptproblem: die Klasse `good_wine = 1` ist selten, daher schwer für Modelle, diese zuverlässig vorherzusagen.
- **Fazit:** Die Modelle sind insgesamt ganz gut, um schlechte Weine zuverlässig zu erkennen, 
  aber das Vorhersagen wirklich guter Weine bleibt relativ schwierig. Am besten abgeschnitten hat hier das
  KNN-Modell, es zeigt die beste Balance zwischen den Metriken.
- **Beantwortung der Fragestellung:** Man kann bis zu einem gewissen Grad anhand der chemischen Eigenschaften vorhersagen, ob ein Wein eine hohe Qualität besitzt oder nicht.